# Ejercicios: NLL bigramas (conteo vs. red), trigramas con red neuronal y SGD

* **Tarea 1:** verificar que la NLL del modelo de bigramas probabilístico (contando) y la del modelo de bigramas con red neuronal son (prácticamente) iguales.
* **Tarea 2:** pasar el modelo de red neuronal de bigramas a **trigramas**: encodear los dos caracteres de entrada con one-hot (40 + 40 = 80 dimensiones) y usar una capa de 40 neuronas con 80 entradas cada una. La NLL tiene que dar mejor que la de bigramas.
* **Tarea 3:** entrenar con **Stochastic Gradient Descent** (minibatches). Lo definimos primero porque lo usamos para entrenar todos los modelos.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Cada línea del archivo es un nombre de ciudad (ya codificado en punycode). Descartamos líneas vacías.
dataset = [d for d in open('data/city_names_puny.txt', 'r').read().split('\n') if d]

# Vocabulario: '*' marca inicio/fin de palabra y va en el índice 0, después todos los caracteres ordenados
charset = ['*'] + sorted(list(set([c for d in dataset for c in d])))
ctoi = {c: i for i, c in enumerate(charset)}   # caracter -> índice
itoc = {i: c for i, c in enumerate(charset)}   # índice -> caracter
nclasses = len(charset)                        # 40
len(dataset), nclasses

## Tarea 3 - Stochastic Gradient Descent (SGD) con minibatches

En clase entrenamos con **todo el dataset en cada paso** (full batch): ~1,7 millones de ejemplos por iteración. Eso tiene dos problemas:
* Es **lento**: cada paso hace el forward y el backward sobre todos los ejemplos.
* Usa **mucha memoria**: el one-hot de todo el dataset de trigramas es una matriz de 1,7M x 80 floats (~550 MB, más los intermedios del forward y el backward). En WSL eso no entra.

**Idea de SGD:** en cada paso tomamos un **minibatch** chico de ejemplos al azar (por ejemplo 512), calculamos la loss y el gradiente **solo sobre ese minibatch** y actualizamos los pesos.
* El gradiente de un minibatch es una **estimación ruidosa** del gradiente real. Ruidosa pero barata: en lugar de pocos pasos "perfectos" hacemos muchos pasos "aproximados" y, en conjunto, convergemos mucho más rápido.
* Como la loss de cada minibatch es ruidosa, para saber cómo va el modelo calculamos la NLL sobre **todo** el dataset al final (en pedazos, para no llenar la memoria).
* **Learning rate decay:** en el último 25% de los pasos dividimos el learning rate por 10 para que el modelo "se asiente" cerca del mínimo en lugar de rebotar alrededor por el ruido.
* Guardamos los **índices** de los caracteres y armamos el one-hot **solo para el minibatch** de cada paso.

Las funciones sirven tanto para bigramas (`X` con 1 columna) como para trigramas (`X` con 2 columnas).

In [ ]:
def encode(X):
    # X: (B, k) índices de los k caracteres de contexto -> (B, 40*k) concatenando los one-hot
    return torch.cat([F.one_hot(X[:, j], num_classes=nclasses) for j in range(X.shape[1])], dim=1).float()

def nll_completa(X, Y, W, chunk=100_000):
    # NLL promedio sobre todo el dataset, calculada en pedazos para no llenar la memoria
    total = 0.0
    with torch.no_grad():
        for i in range(0, len(Y), chunk):
            logits = encode(X[i:i+chunk]) @ W
            counts = logits.exp()
            probs = counts / counts.sum(axis=1, keepdims=True)
            total += -probs[range(len(Y[i:i+chunk])), Y[i:i+chunk]].log().sum().item()
    return total / len(Y)

def entrenar_sgd(X, Y, steps=10_000, batch_size=512, lr=10.0):
    g = torch.Generator().manual_seed(42)
    W = torch.randn((X.shape[1] * nclasses, nclasses), generator=g, requires_grad=True)
    losses = []
    for k in range(steps):
        # minibatch: elegimos batch_size ejemplos al azar
        ix = torch.randint(0, X.shape[0], (batch_size,), generator=g)
        # forward pass (solo sobre el minibatch)
        logits = encode(X[ix]) @ W
        counts = logits.exp()
        probs = counts / counts.sum(axis=1, keepdims=True)   # softmax
        loss = -probs[range(batch_size), Y[ix]].log().mean()
        # backward pass
        W.grad = None
        loss.backward()
        # update con learning rate decay en el último 25%
        paso = lr if k < steps * 0.75 else lr / 10
        W.data += -paso * W.grad
        losses.append(loss.item())
        if k % 2000 == 0:
            print(f'{k:6d} loss minibatch: {loss.item():.4f}')
    return W, losses

## Tarea 1 - NLL del modelo de conteo vs. NLL de la red neuronal

### Dataset de bigramas
Igual que en clase: cada nombre se rodea con `*` al inicio y al final.

In [ ]:
xs, ys = [], []
for d in dataset:
    example = ['*'] + list(d) + ['*']
    for c1, c2 in zip(example, example[1:]):
        xs.append(ctoi[c1])
        ys.append(ctoi[c2])
xs = torch.tensor(xs)
ys = torch.tensor(ys)
len(xs)

### Modelo probabilístico (contando)
Contamos los bigramas, normalizamos cada fila y calculamos la NLL **promedio** sobre todos los bigramas del dataset (incluyendo los `*` de inicio y fin, para usar exactamente los mismos ejemplos que ve la red).

> Ojo: en el notebook probabilístico resuelto la NLL se calcula como **suma** y sin los `*`, por eso da un número enorme (~4 millones). Para comparar con la loss de la red hay que usar el **promedio** sobre los mismos bigramas.

In [ ]:
N = torch.zeros((nclasses, nclasses))
N.index_put_((xs, ys), torch.ones(len(xs)), accumulate=True)   # equivale a: bigram_count[c1, c2] += 1 para cada bigrama

P = N / N.sum(axis=1, keepdims=True)   # sin smoothing: el óptimo de máxima verosimilitud
nll_conteo = -P[xs, ys].log().mean().item()
print(f'NLL bigramas (conteo): {nll_conteo:.4f}')

### Modelo de red neuronal (bigramas)
Una capa de 40 neuronas, 40 entradas (one-hot), sin bias, entrenada con SGD. `xs.unsqueeze(1)` convierte el vector de índices en una matriz de 1 columna, que es lo que espera `encode`.

In [ ]:
X2 = xs.unsqueeze(1)   # (N, 1)
W2, losses2 = entrenar_sgd(X2, ys)
nll_red_bigramas = nll_completa(X2, ys, W2)
print(f'NLL bigramas (red neuronal, dataset completo): {nll_red_bigramas:.4f}')

In [ ]:
print(f'NLL bigramas (conteo):       {nll_conteo:.4f}')
print(f'NLL bigramas (red neuronal): {nll_red_bigramas:.4f}')
print(f'Diferencia:                  {nll_red_bigramas - nll_conteo:.4f}')

**Dan prácticamente igual.** Con one-hot, `xenc @ W` es simplemente elegir la fila `W[c1]`, así que la red aprende una tabla de 40x40 logits. La NLL es mínima cuando `softmax(W[c1])` coincide con la fila normalizada de cuentas, es decir cuando `W.exp()` es proporcional a `N`. El modelo de conteo **ya es** ese mínimo (estimador de máxima verosimilitud), así que la red solo puede acercarse a él.

Lo verificamos comparando las probabilidades que aprendió la red con la matriz `P`:

In [ ]:
P_red = F.softmax(W2.detach(), dim=1)
fila = ctoi['a']
print('P conteo, fila a:', P[fila, 14:20])
print('P red,    fila a:', P_red[fila, 14:20])

## Tarea 2 - Modelo de trigramas con red neuronal

Ahora la entrada son **dos** caracteres `(c1, c2)` y queremos predecir `c3`.
* Cada nombre se rodea con `**` al inicio (dos caracteres de contexto) y `*` al final.
* Encodeamos `c1` y `c2` en one-hot de 40 y los **concatenamos** en un vector de 1 x 80.
* `W` pasa a ser de 80 x 40: 40 neuronas, cada una con 80 entradas.

In [ ]:
xs1, xs2, ys3 = [], [], []
for d in dataset:
    example = ['*', '*'] + list(d) + ['*']
    for c1, c2, c3 in zip(example, example[1:], example[2:]):
        xs1.append(ctoi[c1])
        xs2.append(ctoi[c2])
        ys3.append(ctoi[c3])
xs1 = torch.tensor(xs1)
xs2 = torch.tensor(xs2)
ys3 = torch.tensor(ys3)
X3 = torch.stack([xs1, xs2], dim=1)   # (N, 2): los dos caracteres de contexto

for i in range(6):
    print(itoc[xs1[i].item()], itoc[xs2[i].item()], '->', itoc[ys3[i].item()])

In [ ]:
# Cómo queda la entrada de un ejemplo: one-hot(c1) concatenado con one-hot(c2) = vector de 80
x = encode(X3[:1])
print(x.shape)                  # (1, 80)
print(x[0].nonzero().flatten()) # posición de c1 (0..39) y 40 + posición de c2 (40..79)

In [ ]:
W3, losses3 = entrenar_sgd(X3, ys3)   # W3: 80x40
nll_red_trigramas = nll_completa(X3, ys3, W3)
print(f'NLL trigramas (red neuronal, dataset completo): {nll_red_trigramas:.4f}')

### Comparación final
Como referencia también calculamos el modelo de trigramas **por conteo** (el ejercicio del notebook probabilístico: un tensor de 40x40x40).

In [ ]:
N3 = torch.zeros((nclasses, nclasses, nclasses))
N3.index_put_((xs1, xs2, ys3), torch.ones(len(xs1)), accumulate=True)
P3 = N3 / N3.sum(axis=2, keepdims=True)
nll_conteo_trigramas = -P3[xs1, xs2, ys3].log().mean().item()

print(f'NLL bigramas  (conteo):       {nll_conteo:.4f}')
print(f'NLL bigramas  (red neuronal): {nll_red_bigramas:.4f}')
print(f'NLL trigramas (red neuronal): {nll_red_trigramas:.4f}')
print(f'NLL trigramas (conteo):       {nll_conteo_trigramas:.4f}')

* La red de trigramas da **mejor NLL** que la de bigramas: tener dos caracteres de contexto ayuda a predecir el siguiente.
* No llega a la NLL del trigrama por conteo. Con la entrada concatenada, `encode(x) @ W3 = W3[c1] + W3[40 + c2]`: los logits son la **suma** de un aporte de `c1` y otro de `c2`. Son solo 80x40 = 3.200 parámetros contra 40x40x40 = 64.000 de la tabla, así que la red no puede representar cualquier combinación `(c1, c2)`. Aun así mejora bastante a los bigramas.

### Curva de la loss (SGD)
La loss de cada minibatch es ruidosa (cada paso ve ejemplos distintos). Promediando cada 100 pasos se ve la tendencia. El escalón al 75% es el learning rate decay.

In [ ]:
def suavizar(l, w=100):
    return torch.tensor(l).view(-1, w).mean(1)

plt.figure(figsize=(10, 4))
plt.plot(losses2, alpha=0.2, color='C0')
plt.plot(torch.arange(len(suavizar(losses2))) * 100 + 50, suavizar(losses2), color='C0', label='bigramas')
plt.plot(losses3, alpha=0.2, color='C1')
plt.plot(torch.arange(len(suavizar(losses3))) * 100 + 50, suavizar(losses3), color='C1', label='trigramas')
plt.ylim(2.3, 3.5); plt.xlabel('paso'); plt.ylabel('NLL minibatch'); plt.legend(); plt.grid();

* Cada paso de SGD procesa 512 ejemplos en lugar de 1,7 millones (unas 3.400 veces menos trabajo por paso). Con 10.000 pasos vemos en total ~5 millones de ejemplos, el equivalente a ~3 pasadas (epochs) por el dataset.
* Aun así, el modelo de bigramas llega prácticamente a la NLL óptima del conteo: el ruido del minibatch no impide converger.

### Sampleando del modelo de trigramas

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for _ in range(15):
    c1, c2 = 0, 0   # contexto inicial: '*', '*'
    out = []
    while True:
        x = encode(torch.tensor([[c1, c2]]))
        p = F.softmax(x @ W3, dim=1)
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itoc[ix])
        c1, c2 = c2, ix
    print(''.join(out))